# Hospital Readmission Prediction Within 30 Days

## Objective

In this milestone, I built and compared machine-learning models to predict whether a patient with diabetes would be readmitted to a hospital within 30 days.

I will:
- Load and clean the UCI hospital dataset.
- Create a binary readmission target.
- Split records by patient to reduce data leakage.
- Train Decision Tree and Random Forest classifiers.
- Compare the models against a simple baseline.
- Evaluate precision, recall, F1, ROC-AUC and average precision.
- Visualize feature importance and save the final model.

**Note:** This is a learning project, not a validated clinical prediction tool.

## Dataset

I used the **Diabetes 130-US Hospitals for Years 1999–2008** dataset from the UCI Machine Learning Repository.

It contains hospital records of diabetic patients and the `readmitted` column tells whether a patient was readmitted:

- `<30` means readmitted within 30 days
- `>30` means readmitted after 30 days
- `NO` means not readmitted

Dataset link: https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008

In [ ]:
import io
import zipfile
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree

sns.set_style("whitegrid")
RANDOM_STATE = 42

## 1. Load the dataset

In [ ]:
url = (
    "https://archive.ics.uci.edu/static/public/296/"
    "diabetes+130-us+hospitals+for+years+1999-2008.zip"
)

local_file = Path("uci_diabetes.zip")

if local_file.exists():
    file_content = local_file.read_bytes()
else:
    response = requests.get(url, timeout=180)
    response.raise_for_status()
    file_content = response.content
    local_file.write_bytes(file_content)  # Reuse the download on later runs

with zipfile.ZipFile(io.BytesIO(file_content)) as folder:
    with folder.open("diabetic_data.csv") as file:
        df = pd.read_csv(file)

print("Dataset shape:", df.shape)
df.head()

In [ ]:
df.info()

## 2. Basic data cleaning

In [ ]:
# In this dataset, ? represents a missing value
df = df.replace("?", np.nan)

# Remove patients who died or were discharged to hospice because they are not eligible for readmission
not_eligible = [11, 13, 14, 19, 20, 21]
df = df[~df["discharge_disposition_id"].isin(not_eligible)].copy()

# Create a binary output column
df["readmitted_30_days"] = (df["readmitted"] == "<30").astype(int)

print("Rows after cleaning:", len(df))
print("Missing values:", df.isnull().sum().sum())
df["readmitted_30_days"].value_counts()

### Target distribution

Only a minority of encounters are followed by readmission within 30 days. A model that predicts "No" for everyone could still have high accuracy, so **accuracy alone is misleading**. I will also report precision, recall, F1, ROC-AUC and average precision for the positive (readmitted within 30 days) class.

In [ ]:
target_counts = df["readmitted_30_days"].value_counts().sort_index()

plt.figure(figsize=(6, 4))
plt.bar(["Not readmitted within 30 days", "Readmitted within 30 days"],
        target_counts.values, color=["steelblue", "salmon"])
plt.title("Target Class Distribution")
plt.ylabel("Number of patients")
plt.xticks(rotation=10)
plt.show()

print("30-day readmission rate:", round(df["readmitted_30_days"].mean() * 100, 2), "%")

## 3. Select features

I selected a smaller set of understandable features:

- **Demographic:** race, gender and age
- **Clinical:** A1C result, glucose result, insulin, diabetes medicine and number of diagnoses
- **Hospitalization-related:** time in hospital, lab tests, procedures, medicines and previous hospital visits

Patient and encounter IDs are not used as model features.

In [ ]:
numeric_features = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

categorical_features = [
    "race",
    "gender",
    "age",
    "admission_type_id",
    "admission_source_id",
    "max_glu_serum",
    "A1Cresult",
    "insulin",
    "change",
    "diabetesMed"
]

# These ID columns contain category numbers, so they are converted to text
df["admission_type_id"] = df["admission_type_id"].astype(str)
df["admission_source_id"] = df["admission_source_id"].astype(str)

features = numeric_features + categorical_features

X = df[features]
y = df["readmitted_30_days"]
patient_groups = df["patient_nbr"]

print("Selected features:", len(features))

## 4. Split the data

The data is divided into training, validation and testing sets.

- Training data is used to train the models.
- Validation data is used to compare the models.
- Testing data is used only for the final result.

The split is based on `patient_nbr`. This prevents the same patient from appearing in both training and testing data and reduces data leakage.

In [ ]:
# Split patients into approximately 70% training and 30% temporary data
split1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_STATE)
train_index, temp_index = next(split1.split(X, y, groups=patient_groups))

X_train = X.iloc[train_index]
y_train = y.iloc[train_index]
train_groups = patient_groups.iloc[train_index]

X_temp = X.iloc[temp_index]
y_temp = y.iloc[temp_index]
temp_groups = patient_groups.iloc[temp_index]

# Divide the remaining patients into validation and test groups
split2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=RANDOM_STATE)
val_index, test_index = next(split2.split(X_temp, y_temp, groups=temp_groups))

X_val = X_temp.iloc[val_index]
y_val = y_temp.iloc[val_index]
val_groups = temp_groups.iloc[val_index]

X_test = X_temp.iloc[test_index]
y_test = y_temp.iloc[test_index]
test_groups = temp_groups.iloc[test_index]

# Check that a patient is never present in two datasets
assert set(train_groups).isdisjoint(set(val_groups))
assert set(train_groups).isdisjoint(set(test_groups))
assert set(val_groups).isdisjoint(set(test_groups))

for label, labels in [("Training", y_train), ("Validation", y_val), ("Testing", y_test)]:
    print(f"{label}: {len(labels)} rows, positive rate: {labels.mean():.2%}")
print("Patient group overlap: none")

## 5. Preprocess the data

In [ ]:
numeric_process = Pipeline([
    ("missing_values", SimpleImputer(strategy="median"))
])

categorical_process = Pipeline([
    ("missing_values", SimpleImputer(strategy="most_frequent")),
    ("encoding", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_process, numeric_features),
    ("categorical", categorical_process, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)

## 6. Train the models

`class_weight='balanced'` is used because the readmitted class has fewer records.

In [ ]:
decision_tree = DecisionTreeClassifier(
    max_depth=6,
    min_samples_leaf=100,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

random_forest = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=20,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

decision_tree.fit(X_train_processed, y_train)
random_forest.fit(X_train_processed, y_train)

print("Both models are trained.")

## 7. Compare models on validation data

In [ ]:
def calculate_metrics(model, X_data, y_data):
    predicted = model.predict(X_data)
    probability = model.predict_proba(X_data)[:, 1]

    return {
        "Accuracy": accuracy_score(y_data, predicted),
        "Precision": precision_score(y_data, predicted, zero_division=0),
        "Recall": recall_score(y_data, predicted, zero_division=0),
        "F1 Score": f1_score(y_data, predicted, zero_division=0),
        "ROC-AUC": roc_auc_score(y_data, probability),
        "Avg Precision": average_precision_score(y_data, probability)
    }

validation_results = pd.DataFrame({
    "Decision Tree": calculate_metrics(decision_tree, X_val_processed, y_val),
    "Random Forest": calculate_metrics(random_forest, X_val_processed, y_val)
}).T

validation_results.round(3)


### Simple baseline

I compare my models against a classifier that always predicts the most common class (not readmitted within 30 days). This helps show why accuracy alone is not a useful measure for an imbalanced dataset.


In [ ]:

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train_processed, y_train)

baseline_results = pd.DataFrame({
    "Majority Baseline": calculate_metrics(baseline, X_val_processed, y_val)
}).T

pd.concat([baseline_results, validation_results]).round(3)


I selected the model with the higher **validation F1 score** for the 30-day readmission class. I use validation data for choosing the model and reserve test data for one final evaluation. The baseline is for comparison, not a candidate model.

In [ ]:
best_model_name = validation_results["F1 Score"].idxmax()

if best_model_name == "Decision Tree":
    best_model = decision_tree
else:
    best_model = random_forest

print("Selected model:", best_model_name)

## 8. Final testing

In [ ]:
test_prediction = best_model.predict(X_test_processed)
test_probability = best_model.predict_proba(X_test_processed)[:, 1]

test_metrics = pd.DataFrame({
    "Accuracy": [accuracy_score(y_test, test_prediction)],
    "Precision": [precision_score(y_test, test_prediction, zero_division=0)],
    "Recall": [recall_score(y_test, test_prediction)],
    "F1 Score": [f1_score(y_test, test_prediction)],
    "ROC-AUC": [roc_auc_score(y_test, test_probability)]
}, index=[best_model_name])

test_metrics.round(3)

In [ ]:
print(classification_report(
    y_test,
    test_prediction,
    target_names=["Not readmitted within 30 days", "Readmitted within 30 days"]
))

In [ ]:
matrix = confusion_matrix(y_test, test_prediction)

plt.figure(figsize=(6, 5))
sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted No", "Predicted Yes"],
            yticklabels=["Actual No", "Actual Yes"])
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## 9. Interpret the models

### Decision Tree

The first three levels are displayed so that the main decision rules are readable.

In [ ]:
feature_names = preprocessor.get_feature_names_out()

plt.figure(figsize=(22, 9))
plot_tree(
    decision_tree,
    feature_names=feature_names,
    class_names=["No", "Yes"],
    max_depth=3,
    filled=True,
    rounded=True,
    fontsize=8
)
plt.title("Decision Tree - First Three Levels")
plt.show()

### Random Forest feature importance

I inspected which inputs the Random Forest used most. Built-in tree importance is only a rough measure: it can favor some variables and does **not** prove that a feature causes hospital readmission. One-hot encoded categories also appear as separate features.

In [ ]:
importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": random_forest.feature_importances_
}).sort_values("Importance", ascending=False).head(15)

plt.figure(figsize=(9, 6))
sns.barplot(data=importance, x="Importance", y="Feature", color="seagreen")
plt.title("Top 15 Important Features")
plt.show()

importance

## 10. Save the final model

In [ ]:
# Save both preprocessing and the selected model together
final_system = {
    "preprocessor": preprocessor,
    "model": best_model,
    "features": features,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "positive_label": "Readmitted within 30 days",
    "threshold": 0.5
}

joblib.dump(final_system, "hospital_readmission_model.joblib")
print("Model saved as hospital_readmission_model.joblib")
print("Input note: admission_type_id and admission_source_id should be strings.")

## Conclusion

- I created a binary classification task from hospital encounters for patients with diabetes.
- I used patient-based training, validation and test splits to reduce patient-level data leakage.
- I built preprocessing steps for missing values and categorical data.
- I compared Decision Tree and Random Forest models with a majority-class baseline.
- Because the positive class is uncommon, I evaluated precision, recall, F1, ROC-AUC and average precision, not only accuracy.
- I selected a model using validation F1 and evaluated it on a separate test set.
- I visualized the Decision Tree and Random Forest feature importance.

**Limitations:** This dataset comes from US hospitals in 1999–2008 and contains encounters for patients with diabetes. Results may not generalize to other hospitals or patient groups. Class imbalance and modest precision can cause many false alerts. This model is for learning, not clinical decision-making.
